In [83]:
import pandas as pd

In [115]:
data = pd.read_csv('data/ingredients.csv', sep=',')
display(data)

,id,name,packing,store,price,type
0,1,сахар,1 кг,лента,62.99,продукты
1,2,печенье на декор,1 упаковка,лента,89.99,декор
2,3,сгущенка рогачев,370 мл,лента,164.99,продукты
3,4,малина и голубика в сердце,1 упаковка,перекресток,579.99,декор
4,5,яйцо с1 окское,10 шт,лента,109.99,продукты
...,...,...,...,...,...,...
134,135,трайфлы,100 шт,ольга,1200.00,упаковка
135,136,коробка кап6,1 шт,ольга,60.00,упаковка
136,137,яйцо с0 365 дней,10 штук,лента,99.99,продукты
137,138,малина с/м,1 кг,ольга,1600.00,продукты


In [85]:
data[data['type'] =='продукты']['packing'].value_counts()

packing
1 кг          18
80 г           8
370 мл         8
300 г          6
1 л            6
2 кг           4
10 штук        4
5 л            3
10 шт          3
500 мл         3
260 мл         2
220 г          2
500 г          2
580 мл         2
850 мл         2
10 г           2
50 г           2
350 г          1
75 г           1
2,2 кг         1
10 кг          1
450 г          1
2 г            1
425 мл         1
100 г          1
150 г          1
180 г          1
1,5 г          1
30 шт          1
0.5 л          1
200 мл         1
1 упаковка     1
90 г           1
Name: count, dtype: int64

>Задача №1: получить столбец с ценой за 100г/100мл ингридиента
Для этого: 
* для упаковки "_ кг" привести к цене за 100 г 
* для упаковки "_ л" привести к цене за 100 мл
* для упаковки "_ штук" привести к цене за 1 штуку
* для упаковки "1 упаковка" оставить как есть
* для упаковки "_ кг" привести к цене за 100 г

In [116]:
def to_gramm(string):
    string = string.split(' ')

    if string[1] == 'кг':
        if ',' in string[0]:
            return float(string[0].replace(',','.')) * 1000
        else:
            return float(string[0]) * 1000
    elif string[1] == 'л':
            if ',' in string[0]:
                return float(string[0].replace(',','.')) * 1000
            else:
                return float(string[0]) * 1000
    else:
        if ',' in string[0]:
            return float(string[0].replace(',','.'))
        else:
            return float(string[0])

In [117]:
to_gramm('1 упаковка')

1.0

In [118]:
def units(string):
    string = string.split(' ')

    if string[1] == 'кг' or string[1] == 'г':
        return 'г'
    elif string[1] == 'л' or string[1] == 'мл':
        return 'мл'
    elif string[1] == 'штук' or string[1] == 'шт' or string[1] == 'лист':
        return 'шт'
    elif string[1] == 'упаковка':
        return 'уп'
    else:
        pass


In [119]:
data['units'] = data['packing'].apply(units)

In [120]:
data['package weight'] = data['packing'].apply(to_gramm)

In [121]:
data['price for unit'] = 0
data.head(3)

,id,name,packing,store,price,type,units,package weight,price for unit
0,1,сахар,1 кг,лента,62.99,продукты,г,1000.0,0
1,2,печенье на декор,1 упаковка,лента,89.99,декор,уп,1.0,0
2,3,сгущенка рогачев,370 мл,лента,164.99,продукты,мл,370.0,0


In [122]:
mask1 = (data['units'] == 'шт') | (data['units'] == 'уп')
mask2 = (data['units'] == 'г') | (data['units'] == 'мл')

In [123]:
data.loc[mask1,'price for unit'] = data.loc[mask1, 'price'] / data.loc[mask1,'package weight']
data.loc[mask2,'price for unit'] = (100 / data.loc[mask2, 'package weight']) * data.loc[mask2,'price']

In [124]:
data = data.drop('packing', axis = 1)

In [125]:
data['3_choco_product'] = 0

In [126]:
data

,id,name,store,price,type,units,package weight,price for unit,3_choco_product
0,1,сахар,лента,62.99,продукты,г,1000.0,6.299000,0
1,2,печенье на декор,лента,89.99,декор,уп,1.0,89.990000,0
2,3,сгущенка рогачев,лента,164.99,продукты,мл,370.0,44.591892,0
3,4,малина и голубика в сердце,перекресток,579.99,декор,уп,1.0,579.990000,0
4,5,яйцо с1 окское,лента,109.99,продукты,шт,10.0,10.999000,0
...,...,...,...,...,...,...,...,...,...
134,135,трайфлы,ольга,1200.00,упаковка,шт,100.0,12.000000,0
135,136,коробка кап6,ольга,60.00,упаковка,шт,1.0,60.000000,0
136,137,яйцо с0 365 дней,лента,99.99,продукты,шт,10.0,9.999000,0
137,138,малина с/м,ольга,1600.00,продукты,г,1000.0,160.000000,0


На этом этапе задача выполнена. Далее будет идти подготовка к объединению с таблицей рецепта


---

В таблице с ингридиентами  название товаров записаны как в чеке. В работе используется короткое название.
Соответственно необходимо произвести предобработку, чтобы привести название в рабочее короткое название

In [95]:
'''
def rename(string):
    string = string.split(' ')
    if len(string) == 1:
        return ' '.join(string[:2])
    else:
        return ' '.join(string[:2])
#rename('шоколад белый воздушный 80 г')
'''

"\ndef rename(string):\n    string = string.split(' ')\n    if len(string) == 1:\n        return ' '.join(string[:2])\n    else:\n        return ' '.join(string[:2])\n#rename('шоколад белый воздушный 80 г')\n"

In [96]:
#data['work_name'] = data['name'].apply(rename)

In [97]:
'''
mask3 = data['type']=='продукты'
mask4 = (data['store'] == 'лента') | (data['store'] == 'ольга')
#mask5 = data['store'] == 'ольга'
table_product = data[mask3 & mask4]
'''

"\nmask3 = data['type']=='продукты'\nmask4 = (data['store'] == 'лента') | (data['store'] == 'ольга')\n#mask5 = data['store'] == 'ольга'\ntable_product = data[mask3 & mask4]\n"

In [127]:
recept_3_choco = pd.read_csv('data/3 шоколада.csv', sep = ',')

In [128]:
recept_3_choco

,product_name,weight
0,яйцо c0,2
1,сахар,60
2,мука,53
3,разрыхлитель,3
4,какао,15
5,шоколад темный,275
6,шоколад молочный,275
7,шоколад белый,275
8,сливки 33%,825
9,желатин,30


Идея какая:

0. запускаем цикл for по столбцу "product_name"
1. обращаемся к 1 ячейке "product_name" таблицы recept_3_choco, сохраняем в переменную Х
2. в таблице игридиентов Запускаем цикл enumerate по ячейке столбца "name" и индексу.
обращаемся к ячейкам столбца 'name', с помощью re ищем вхождение переменной в строке.
3. Создаем новый столбец, в который по индексу заносим Nan, если совпадения нет, и переменную Х, если есть
4. Итоговый результат - в таблице игридиентов новый столбец, в котором проставлены имена продуктов в том же виде, как в таблице 3 шоколада. 

In [ ]:
import re

for values in recept_3_choco['product_name']:
    for string in  data['name']:
        pattern = re.compile(values)
        result = str(pattern.findall(string))
        #теперь нужно присвоить значение столбцу
        #data[string, '3_choco_product'] = result

---
объединения таблиц

In [100]:
'''price_for_3choco = recept_3_choco.merge(
    table_product,
    left_on='product_name',
    right_on = 'work_name',
    how='left'
)
'''

"price_for_3choco = recept_3_choco.merge(\n    table_product,\n    left_on='product_name',\n    right_on = 'work_name',\n    how='left'\n)\n"

In [101]:
#price_for_3choco

яйца потерял :(

Вопрос: как выбрать теперь товары. Идея вот какая. 1. самый дешевый вариант торта - выбираем при прочих равных минимальную цену за 100 г. 2. недешвый вариант, но качественный

идея. создать признак - мигалку. если одинаковые значение df['name'] мы ищем минимум среди соответстующих df['price']. создаем в столбце True
далее группируемся и считаем сумму

In [102]:
def choice(*arg):
    '''
    Логика работы: проходим в цикле по столбцу "work_name", уникальные значения сразу становятся True в признаке-мигалке.
    Если значения повторяются, находим минимальное значения "price for unit" и эту строку присваиваем True в признаке-мигалке.
    Результат работы функции - создание признака со значениями True либо в уникальных строках с "work_name", либо где значение 
    соответствующего  "price for unit" минимально.
    
    шаг 1: проверка значений на уникальность
    '''
    
    pass

In [103]:
price_for_3choco['work_name']

0                  NaN
1                сахар
2                  NaN
3                  NaN
4                какао
5       шоколад темный
6       шоколад темный
7       шоколад темный
8     шоколад молочный
9     шоколад молочный
10    шоколад молочный
11       шоколад белый
12       шоколад белый
13          сливки 33%
14                 NaN
15                 NaN
Name: work_name, dtype: object